In [1]:
!pip install -q diffusers accelerate transformers gradio Pillow xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 43.0 MB/s eta 0:00:00


In [2]:
import torch
from PIL import Image, ImageDraw, ImageOps
import gradio as gr
from diffusers import (
    StableDiffusionControlNetPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
)

# ======================================================================
# 1. Model Setup (ControlNet Scribble + Stable Diffusion 1.5)
# ======================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Booting Base Generation Models on {device}...")

try:
    controlnet_model = ControlNetModel.from_pretrained(
        "lllyasviel/control_v11p_sd15_scribble",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    )

    controlnet_pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet_model,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        safety_checker=None,
    )

    controlnet_pipe.scheduler = UniPCMultistepScheduler.from_config(controlnet_pipe.scheduler.config)

    if device == "cuda":
        controlnet_pipe = controlnet_pipe.to(device)
        controlnet_pipe.enable_xformers_memory_efficient_attention()

except Exception as e:
    print(f"ControlNet Error: {e}")

# ======================================================================
# 2. Simplified Mask Extraction
# ======================================================================
def extract_drawing(data):
    """
    Simplified extraction: Flattens the canvas into a single stroke layer.
    If drawing exists, it extracts it as white-on-black.
    If no drawing exists, it assumes an uploaded paper sketch and inverts it.
    """
    if data is None or not isinstance(data, dict):
        return None, None

    bg_raw = data.get("background")
    layers = data.get("layers", [])

    # 1. Handle Background
    if bg_raw is None:
        bg = Image.new("RGB", (512, 512), (255, 255, 255))
    else:
        if bg_raw.mode in ('RGBA', 'LA') or (bg_raw.mode == 'P' and 'transparency' in bg_raw.info):
            bg = Image.new("RGB", bg_raw.size, (255, 255, 255))
            bg.paste(bg_raw, mask=bg_raw.split()[-1])
        else:
            bg = bg_raw.convert("RGB")

    # 2. Flatten all strokes into a single layer
    has_drawing = False
    stroke = Image.new("RGBA", bg.size, (0, 0, 0, 0))
    for layer in layers:
        if layer is not None:
            stroke.alpha_composite(layer.convert("RGBA"))
            # Check if user actually drew anything
            if stroke.split()[3].getextrema()[1] > 0:
                has_drawing = True

    # 3. Create the extracted B&W Mask
    sketch = Image.new("L", bg.size, 0) # Base pure black

    if has_drawing:
        # User sketched directly on the canvas
        alpha = stroke.split()[3]
        white_stroke = Image.new("L", bg.size, 255)
        sketch.paste(white_stroke, mask=alpha)
    else:
        # User uploaded a sketch but didn't draw. Invert the image.
        gray = bg.convert("L")
        sketch = ImageOps.invert(gray)
        # Clean up light gray noise to pure black
        sketch = sketch.point(lambda p: 255 if p > 80 else 0)

    return bg, sketch

def make_text_image(msg: str, size=(512, 512)):
    img = Image.new("RGB", size, color=(255, 100, 100))
    draw = ImageDraw.Draw(img)
    draw.text((20, 256), msg, fill=(0, 0, 0))
    return img

# ======================================================================
# 3. Core Generation Logic
# ======================================================================
def run_controlnet(gr_input, prompt):
    bg, sketch = extract_drawing(gr_input)
    if bg is None:
        err = make_text_image("Error: Canvas is empty.")
        return err, err

    # Force 512x512 resolution for stability
    scribble_mask = sketch.convert("RGB").resize((512, 512), Image.LANCZOS)

    try:
        img = controlnet_pipe(
            prompt=prompt,
            negative_prompt="low quality, messy, faces, ugly, bad proportions, distorted",
            image=scribble_mask,
            num_inference_steps=30,
            guidance_scale=7.5
        ).images[0]
    except Exception as e:
        err = make_text_image(f"Generation Error:\n{str(e)}")
        return err, scribble_mask

    return img, scribble_mask

# ======================================================================
# 4. Professional Gradio UI Layout (Dark Theme + Status Indicator)
# ======================================================================

# High-end Custom CSS for a Dark SaaS-like experience
custom_css = """
footer {visibility: hidden !important;}
/* 1. Black/Dark Theme Backgrounds */
.gradio-container {font-family: 'Inter', system-ui, sans-serif !important; background-color: #0f172a !important; color: #e2e8f0 !important;}
/* 2. Smaller Nav Bar */
#header-container {text-align: center; margin-bottom: 15px; padding: 15px 20px; background: linear-gradient(135deg, #000000, #1e1b4b); color: white; border-radius: 12px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.5);}
#header-container h1 {font-weight: 800; font-size: 2em; margin-bottom: 5px; color: white !important;}
#header-container p {font-size: 1em; color: #a5b4fc; opacity: 0.9; margin-top: 0;}
/* Dark Mode Panel Cards */
.panel-card {border: 1px solid #334155 !important; border-radius: 12px; background: #1e293b !important; padding: 20px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.3);}
.section-title {font-weight: 700; color: #f8fafc !important; font-size: 1.2em; margin-bottom: 15px; border-bottom: 2px solid #334155; padding-bottom: 10px;}
/* Status Text Styling */
.status-text {text-align: center; font-weight: 600; color: #fbbf24 !important; margin-top: 10px;}
"""

with gr.Blocks(theme=gr.themes.Base(), css=custom_css) as demo:

    # --- HEADER ---
    gr.HTML("""
    <div id="header-container">
        <h1>✨ SketchControl-AI Studio</h1>
        <p>Professional Sketch-to-Image Synthesis</p>
    </div>
    """)

    # --- ROW 1: EDITING & CONTROL (Inputs) ---
    with gr.Row():

        # Canvas Column
        with gr.Column(scale=2, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>🖍️ 1. Studio Canvas</div>")
            raw_sketch_input = gr.ImageEditor(
              label="Upload a Sketch OR Draw Here",
              type="pil",
              interactive=True,
              height=450,
              # Kept the brush black so you can see it on the white canvas!
              brush=gr.Brush(colors=["#000000"], color_mode="fixed", default_size=4)
            )

        # Prompt & Generation Controls Column
        with gr.Column(scale=1, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>📝 2. Art Direction</div>")
            prompt_input = gr.Textbox(
                label="Scene Prompt",
                placeholder="E.g., A highly detailed futuristic city, cinematic lighting, 8k resolution...",
                value="A highly detailed landscape masterpiece, lush mountains, clear lake, cinematic lighting",
                lines=5
            )

            gr.Markdown("<br>") # Spacing
            gen_btn = gr.Button("🚀 Generate Render", variant="primary", size="lg")

            # ✨ NEW: Processing indicator explicitly placed below the button
            status_indicator = gr.Markdown("", elem_classes="status-text")

    # --- ROW 2: RESULTS (Outputs) ---
    gr.HTML("<br><div class='section-title' style='text-align:center; background:#1e293b; padding:15px; border-radius:12px; border: 1px solid #334155; color: white;'>📊 Output Analytics</div>")

    with gr.Row():

        # Mask Vision Column
        with gr.Column(scale=1, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>🔍 AI Structure Vision</div>")
            gr.Markdown("<span style='color: #94a3b8; font-size: 0.9em;'>This pure black-and-white mask is the structural blueprint extracted from your canvas that the AI follows.</span>")
            mask_output = gr.Image(label="", type="pil", height=450, interactive=False)

        # Final Render Column
        with gr.Column(scale=1, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>🖼️ Final Render</div>")
            gr.Markdown("<span style='color: #94a3b8; font-size: 0.9em;'>The final synthesized image created by combining your text prompt and the structure vision (or sketch).</span>")
            base_output = gr.Image(label="", type="pil", height=450, interactive=False, show_download_button=True)



    gen_btn.click(
        fn=lambda: "⏳ Generating image... Please wait.",
        inputs=None,
        outputs=status_indicator,
        queue=False
    ).then(
        fn=run_controlnet,
        inputs=[raw_sketch_input, prompt_input],
        outputs=[base_output, mask_output]
    ).then(
        fn=lambda: "✅ Generation Complete!",
        inputs=None,
        outputs=status_indicator,
        queue=False
    )

demo.launch(debug=True, share=True, show_api=False)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Booting Base Generation Models on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet.StableDiffusionControlNetPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://59a8429cf6f2de3fcd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  0%|          | 0/30 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://59a8429cf6f2de3fcd.gradio.live
